# Пример использования Iceberg Metastore с JupyterHub

In [79]:
pip list 

Package                   Version
------------------------- --------------
aiobotocore               2.24.2
aiohappyeyeballs          2.6.1
aiohttp                   3.12.15
aioitertools              0.12.0
aiosignal                 1.4.0
alembic                   1.14.0
annotated-types           0.7.0
anyio                     4.8.0
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.4
attrs                     24.3.0
babel                     2.16.0
beautifulsoup4            4.12.3
bleach                    6.2.0
boto3                     1.40.18
botocore                  1.40.18
cachetools                5.5.2
certifi                   2024.12.14
certipy                   0.2.1
cffi                      1.17.1
charset-normalizer        3.4.1
click                     8.2.1
comm                      0.2.2
cryptography              44.0.0
debugpy                   1.8.11
decorat

### Необходимые переменные

In [ ]:
uri = "postgresql://<user>:<password>@<ip_db>/<database_name>"
s3_bucket_name = "<s3_bucket_name>"
db_name = "<db_name>"  # которую указывали при создании metastore
ENDPOINT = "https://hb.bizmrg.com"
ACCESS_KEY = "<ACCESS_KEY>"
SECRET_KEY = "<SECRET_KEY>"

### Создание namespace в Iceberg

In [181]:
from pyiceberg.catalog import load_catalog
 
catalog = load_catalog(
    "my_catalog",
    **{
        "type": "sql",
        "uri": uri,
        "warehouse": f"s3://{s3_bucket_name}/datatest123"
    }
)

try:
    catalog.create_namespace(db_name)
    print(f"Namespace {db_name} создан успешно!")
except Exception as e:
    print(f"Ошибка при создании namespace: {e}")

Namespace metastore создан успешно!


 ### Вывести все неймспейсы в Iceberg

In [182]:
catalog.list_namespaces()

[('metastore',)]

### Вывести все таблицы

In [183]:
catalog.list_tables(db_name)

[]

### Проверка подключения к S3

In [184]:
import boto3

try:
    s3_client = boto3.client(
        's3',
        endpoint_url=ENDPOINT,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY
    )
     
    response = s3_client.list_objects_v2(Bucket=s3_bucket_name, MaxKeys=1)
    print(f"Подключение к S3 успешно. Bucket: {s3_bucket_name}")
     
    if 'Contents' in response:
        print("Bucket не пустой")
        print("Первые объекты:")
        for obj in response['Contents'][:3]:
            print(f"  - {obj['Key']}")
    else:
        print("Bucket пустой")
         
except Exception as e:
    print(f"Ошибка подключения к S3: {e}")

Подключение к S3 успешно. Bucket: jh-test-iceberg
Bucket не пустой
Первые объекты:
  - datatest123/


In [174]:
import os
import boto3
import pyarrow as pa
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, IntegerType
import pandas as pd
from typing import Union

### Функция по приведению типов данных pandas DataFrame для совместимости с Iceberg

In [150]:
def prepare_dataframe_for_iceberg(data_df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит типы данных pandas DataFrame для совместимости с Iceberg
     
    Args:
        data_df: Исходный pandas DataFrame
         
    Returns:
        DataFrame с приведенными типами данных
    """
    prepared_df = data_df.copy()
     
    for col in prepared_df.columns:
        dtype = prepared_df[col].dtype
         
        if dtype == 'object':
            prepared_df[col] = prepared_df[col].astype('string')
         
        elif pd.api.types.is_integer_dtype(dtype):
            # Конвертируем все int типы в int64 для совместимости с Iceberg
            prepared_df[col] = prepared_df[col].astype('int64')
  
        elif pd.api.types.is_float_dtype(dtype):
            prepared_df[col] = prepared_df[col].astype('float64')
         
 
        elif pd.api.types.is_bool_dtype(dtype):
            prepared_df[col] = prepared_df[col].astype('boolean')
         
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            prepared_df[col] = prepared_df[col].astype('datetime64[us]')
     
    return prepared_df
 

### Функция для создания PyArrow схемы

In [151]:
def create_pyarrow_scheme(prepared_df: pd.DataFrame) -> pa.schema:
    fields = []
    for col in prepared_df.columns:
        if pd.api.types.is_datetime64_any_dtype(prepared_df[col].dtype):
            fields.append(pa.field(col, pa.timestamp('us')))
        elif prepared_df[col].dtype == 'string':
            fields.append(pa.field(col, pa.string()))
        elif prepared_df[col].dtype == 'int64':
            fields.append(pa.field(col, pa.int64()))
        elif prepared_df[col].dtype == 'float64':
            fields.append(pa.field(col, pa.float64()))
        elif prepared_df[col].dtype == 'boolean':
            fields.append(pa.field(col, pa.bool_()))
        else:
            fields.append(pa.field(col, pa.from_pandas_dtype(prepared_df[col].dtype)))
    
    return pa.schema(fields)

### Функция добавления pandas DataFrame в Iceberg таблицу

In [153]:
from pyiceberg.table import Table
import pandas as pd
import pyarrow as pa

def add_data_to_iceberg_table(table: Table, data_df: pd.DataFrame) -> None:
    """
    Добавляет pandas DataFrame в Iceberg таблицу

    Args:
        table: Iceberg таблица
        data_df: pandas DataFrame для добавления
    """
    # Подготавливаем данные для Iceberg
    prepared_df = prepare_dataframe_for_iceberg(data_df)
    
    # Создаем PyArrow схему
    schema = create_pyarrow_scheme(prepared_df)
    
    arrow_table = pa.Table.from_pandas(prepared_df, schema=schema)

    table.append(arrow_table)

    print(f"Добавлено {len(data_df)} записей в таблицу")
    

### Создание каталога

Можем указать свою папку в бакете в строке warehouse=f"s3://{s3_bucket_name}/datatest123/test_table"

In [ ]:
# Создаем каталог
catalog = load_catalog(
    "my_catalog",
    **{
        "type": "sql",
        "uri": uri,
        "warehouse": f"s3://{s3_bucket_name}/",
        "s3.endpoint": ENDPOINT,
        "s3.access-key-id": ACCESS_KEY,
        "s3.secret-access-key": SECRET_KEY
    }
)

### Удаляем существующую таблицу

In [164]:
# Удаляем существующую таблицу
try:
    catalog.drop_table("metastore.test_table")
    print("Старая таблица удалена")
except:
    print("Старая таблица не найдена")

Старая таблица удалена


### Создаем тестовые данные

In [176]:
test_data = pd.DataFrame({
    'id': [1, 2, 3], 
    'name': ['Alice', 'Bob', 'Charlie'],
    'price': [10.5, 20.3, 30.1],
    'is_active': [True, False, True],
    'created_at': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-03'])
})

### Определяем схему таблицы

In [186]:
from pyiceberg.types import NestedField, StringType, LongType, DoubleType, BooleanType, TimestampType


schema = Schema(
    NestedField(1, "id", LongType(), required=False),
    NestedField(2, "name", StringType(), required=False),
    NestedField(3, "price", DoubleType(), required=False),
    NestedField(4, "is_active", BooleanType(), required=False),
    NestedField(5, "created_at", TimestampType(), required=False)
)

### Создаем таблицу

Можем указать свою папку в бакете в строке location=f"s3://{s3_bucket_name}/datatest123/test_table"

In [187]:
try:
    table = catalog.create_table(
        "metastore.test_table",
        schema=schema,
        location=f"s3://{s3_bucket_name}/metastore/test_table"
    )
    print("Таблица 'metastore.test_table' создана успешно!")
 
except Exception as e:
    print(f"Ошибка при создании таблицы: {e}")

Таблица 'metastore.test_table' создана успешно!


### Добавляем данные

In [159]:
try:
    add_data_to_iceberg_table(table, test_data)
except Exception as e:
    print(f"Ошибка при создании таблицы: {e}")

Добавлено 3 записей в таблицу


### Читаем все данные из таблицы

In [160]:
df_all = table.scan().to_pandas()
print("Все данные в таблице:")
print(df_all)

Все данные в таблице:
   id     name  price  is_active created_at
0   1    Alice   10.5       True 2024-01-01
1   2      Bob   20.3      False 2024-01-02
2   3  Charlie   30.1       True 2024-01-03


### Добавляем новые данные

In [161]:
new_data = pd.DataFrame({
    'id': [4, 5, 6],
    'name': ['Diana', 'Eve', 'Frank'],
    'price': [40.5, 50.3, 60.1],
    'is_active': [False, True, False],
    'created_at': pd.to_datetime(['2024-01-04', '2024-01-05', '2024-01-06'])
})
 
add_data_to_iceberg_table(table, new_data)

Добавлено 3 записей в таблицу


### Читаем все данные из таблицы

In [162]:
df_all = table.scan().to_pandas()
print("Все данные в таблице:")
print(df_all)

Все данные в таблице:
   id     name  price  is_active created_at
0   4    Diana   40.5      False 2024-01-04
1   5      Eve   50.3       True 2024-01-05
2   6    Frank   60.1      False 2024-01-06
3   1    Alice   10.5       True 2024-01-01
4   2      Bob   20.3      False 2024-01-02
5   3  Charlie   30.1       True 2024-01-03
